In [12]:
import torch
import torch.nn as nn

In [13]:
class LayerNorm(nn.Module):
    def __init__(self, dim, eps = 1e-5):
        super().__init__()
        self.eps = eps

        ones = torch.ones((dim,))
        zeros = torch.ones((dim,))
        self.a = nn.Parameter(ones)
        self.b = nn.Parameters(zeros)

    def forward(self, x):
        m = torch.mean(x, dim=-1)
        var = torch.var(x, dim =-1)
        x = (x - m) / torch.sqrt(var + self.eps)
        return self.a * x + self.b
        


In [14]:
cfg_template = {
    "d_model": 768,
    "n_head": 8,
    "qkv_bias": False,
    "context_size": 10000,
    "vocab_size": 50258,
    "dropout": 0.5,
    "transformer_block": 2,
    "temperature": 1.2,
}

In [15]:
class MaskedMultiHeadAttention(nn.Module):
    def __init__(self, cfg):
        assert cfg["d_model"] % cfg["n_head"] == 0, "the n_head must eb a dividor of d_model"

        self.d_in = cfg["d_model"]
        self.d_out = cfg["d_model"]
        self.n_head = cfg["n_head"]
        self.d_head = cfg["d_model"] // cfg["n_head"]

        self.W_q = nn.Linear(cfg["d_model"], cfg["d_model"], bias = cfg["qkv_bias"])
        self.W_k = nn.Linear(cfg["d_model"], cfg["d_model"], bias = cfg["qkv_bias"])
        self.W_v = nn.Linear(cfg["d_model"], cfg["d_model"], bias = cfg["qkv_bias"])
        self.W_o = nn.Linear(cfg["d_model"], cfg["d_model"])

        self.dropout = nn.Dropout(cfg["dropout"])

        mask = torch.tril(torch.ones((cfg["context_size"], cfg["context_size"])))
        self.register_buffer("mask", mask)

    def forward(self, x):

        batch, seq_size, d_model = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(batch, seq_size, self.n_head, self.d_head)
        K = K.view(batch, seq_size, self.n_head, self.d_head)
        V = V.view(batch, seq_size, self.n_head, self.d_head)

        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        attention_weights = Q @ K.transpose(2,3)
        attention_weights = torch.softmax(attention_weights / torch.sqrt(d_model), dim=-1)

        mask = self.mask[:seq_size][:seq_size]

        attention_weights = attention_weights.masked_fill_(mask, -torch.inf)
        attention_weights = self.dropout(attention_weights)
        attention_score = (attention_weights @ V).transpose(1,2)

        attention_score = attention_score.contiguous().view(batch, seq_size, d_model)

        return self.W_o(x)
        

In [16]:
class MultiHeadAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg["d_model"] % cfg["n_head"] == 0, "the number of head must be a dividor of embedding size"

        self.d_head = cfg["d_model"] // cfg["n_head"]
        self.n_head = cfg["n_head"]
        self.d_in = cfg["d_model"]
        self.d_out = cfg["d_model"]

        self.dropout = nn.Dropout(cfg["dropout"])

        self.W_q = nn.Linear(cfg["d_model"], cfg["d_model"], bias=cfg["qkv_bias"])
        self.W_k = nn.Linear(cfg["d_model"], cfg["d_model"], bias=cfg["qkv_bias"])
        self.W_v = nn.Linear(cfg["d_model"], cfg["d_model"], bias=cfg["qkv_bias"])
        self.W_o = nn.Linear(cfg["d_model"], cfg["d_model"])

    def forward(self, x):
        batch, seq_size, d_model = x.shape

        # dim: (b, seq_size, d_model)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # dim: (b, n_head, seq_size, d_head)
        Q = Q.view(batch, seq_size, self.n_head, self.d_head).transpose(1,2)
        K = K.view(batch, seq_size, self.n_head, self.d_head).transpose(1, 2)
        V = V.view(batch, seq_size, self.n_head, self.d_head).transpose(1,2)

        K = K.transpose(2, 3)

        att_weights = Q @ K
        att_weights = torch.softmax(att_weights / torch.sqrt(d_model), dim=-1)

        att_weights = self.dropout(att_weights)

        att_score = (att_weights @ V).transpose(1, 2)
        att_score = att_score.contiguous().view(batch, seq_size, d_model)

        return self.W_o(att_score)






In [17]:
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, cfg, encoder_output):
        super().__init__()
        assert cfg["d_model"] % cfg["n_head"] == 0, "the number of head must be a dividor of embedding size"

        self.d_head = cfg["d_model"] // cfg["n_head"]
        self.n_head = cfg["n_head"]
        self.d_in = cfg["d_model"]
        self.d_out = cfg["d_model"]

        self.dropout = nn.Dropout(cfg["dropout"])

        self.W_q = nn.Linear(cfg["d_model"], cfg["d_model"], bias=cfg["qkv_bias"])
        self.W_k = nn.Linear(cfg["d_model"], cfg["d_model"], bias=cfg["qkv_bias"])
        self.W_v = nn.Linear(cfg["d_model"], cfg["d_model"], bias=cfg["qkv_bias"])
        self.W_o = nn.Linear(cfg["d_model"], cfg["d_model"])

        self.encoder_output = encoder_output

    def forward(self, x, encoder_x):
        batch, seq_size, d_model = x.shape

        # dim: (b, seq_size, d_model)
        Q = self.W_q(x)
        K = self.W_k(encoder_x)
        V = self.W_v(encoder_x)

        # dim: (b, n_head, seq_size, d_head)
        Q = Q.view(batch, seq_size, self.n_head, self.d_head).transpose(1,2)
        K = K.view(batch, seq_size, self.n_head, self.d_head).transpose(1, 2)
        V = V.view(batch, seq_size, self.n_head, self.d_head).transpose(1,2)

        K = K.transpose(2, 3)

        att_weights = Q @ K
        att_weights = torch.softmax(att_weights / torch.sqrt(d_model), dim=-1)

        att_weights = self.dropout(att_weights)

        att_score = (att_weights @ V).transpose(1, 2)
        att_score = att_score.contiguous().view(batch, seq_size, d_model)

        return self.W_o(att_score)

In [18]:
class Encoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.multi_head_attention = MultiHeadAttention(cfg)
        self.norm1 = LayerNorm(cfg["d_model"])
        self.norm2 = LayerNorm(cfg["d_model"])
        self.ff = nn.Linear(cfg["d_model"], cfg["d_model"])

    def forward(self, x):
        shortcut = x
        x = self.multi_head_attention(x)
        x = x + shortcut 
        x = self.norm1(x)

        shortcut = x
        x = self.ff(x)
        x = x + shortcut(x)
        x = self.norm2(x)
        



In [ ]:
class Decoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.d_model = cfg["d_model"]
        self.multi_head_attention = MaskedMultiHeadAttention(cfg)
        self.multi_head_cross_attention = MultiHeadCrossAttention(cfg)
        self.norm1 = LayerNorm(cfg["d_model"])
        self.norm2 = LayerNorm(cfg["d_model"])
        self.norm3 = LayerNorm(cfg["d_model"])
        self.ff = nn.Linear(cfg["d_model"], cfg["d_model"])
        

    def forward(self, x, encoder_x):
        shortcut = self.multi_head_attention(x)
        x = x + shortcut
        x = self.norm1(x)

        shortcut = x
        x = self.multi_head_cross_attention(x, encoder_x)
        x = x + shortcut
        x = self.norm2(x)

        shortcut = x
        x = self.ff(x)
        x = x + shortcut

        return self.norm3(x)


        
        

In [20]:
def _init_positional_encoding(context_size, d_model):
    pos = torch.arange(context_size, dtype=torch.float32).unsqueeze(1)
    div = 10000**(torch.arange(0, d_model, step=2, dtype=torch.float32) / d_model)

   

    pe = torch.zeros((context_size, d_model))
    pe[:, 0::2] = torch.sin(pos/div)
    pe[:, 1::2] = torch.cos(pos/div)

    return pe



In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        self.embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.temp = cfg["temperature"]
        self.register_buffer("pe", _init_positional_encoding(cfg["context_size"], cfg["d_model"]))
        self.encoder = Encoder(cfg)
        self.decoder = Decoder(cfg)
        self.softmax = nn.Softmax()
        self.linear = nn.Linear(cfg["d_model"], cfg["d_model"])

    def forward(self, x):
        _, seq_size, _ = x.shape
        x = self.embedding(x)
        x += self.pe[:seq_size, :]
        encoder_x = self.encoder(x)
        x = self.decoder(x, encoder_x)
        x = self.linear(x)
        x = self.softmax(x / self.temp)


